<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M11/M11_Lab2_LLM_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">📏 M11 Lab 2 — LLM Evaluation Framework</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐⭐ Difficulty: Advanced &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Build a <b>test suite</b> with ground-truth answers</li>
    <li>Implement evaluation metrics: <b>exact match, BLEU, cosine similarity</b></li>
    <li>Run models through the test suite and <b>score each answer</b></li>
    <li>Compare <b>GPT-4.1-mini vs Gemini</b> on the same benchmark</li>
    <li><b>Visualize</b> evaluation results with professional charts</li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Add both `OPENAI_API_KEY` and `GEMINI_API_KEY` to Colab Secrets.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai google-generativeai numpy matplotlib
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, compare_responses, quiz

client = setup_openai()

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Configure Gemini
import google.generativeai as genai
from google.colab import userdata
try:
    genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
    gemini_model = genai.GenerativeModel("gemini-2.5-flash")
    gemini_available = True
    print("✅ Both OpenAI and Gemini configured")
except Exception:
    gemini_available = False
    print("✅ OpenAI configured | ⚠️ Gemini not available (add GEMINI_API_KEY for comparison)")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Why Evaluate LLMs?</h2>
</div>

"Which model is better?" is the wrong question. The right question is: **"Which model is better *for my specific task*?"**

Evaluation helps you:
- **Compare models** objectively (not just vibes)
- **Detect regressions** when switching model versions
- **Justify decisions** to stakeholders ("GPT scored 87% vs Gemini's 82% on our test suite")
- **Identify weaknesses** in specific categories

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Create a Test Suite</h2>
</div>

A test suite is a set of questions with **ground-truth answers**. We'll test across different categories.

In [ ]:
# ============================================================
# 📋 Test Suite: 10 Questions with Ground Truth
# ============================================================

TEST_SUITE = [
    {
        "id": 1,
        "category": "Factual",
        "question": "What is the capital of France? Answer in one word.",
        "ground_truth": "Paris"
    },
    {
        "id": 2,
        "category": "Factual",
        "question": "What programming language is PyTorch written in? Answer in one word.",
        "ground_truth": "Python"
    },
    {
        "id": 3,
        "category": "Math",
        "question": "What is 15% of 2400? Answer with just the number.",
        "ground_truth": "360"
    },
    {
        "id": 4,
        "category": "Math",
        "question": "If a product costs $80 and is discounted 25%, what is the final price? Answer with just the number.",
        "ground_truth": "60"
    },
    {
        "id": 5,
        "category": "Definition",
        "question": "Define 'overfitting' in machine learning in one sentence.",
        "ground_truth": "Overfitting occurs when a model learns the training data too well, including noise and outliers, resulting in poor performance on new unseen data."
    },
    {
        "id": 6,
        "category": "Definition",
        "question": "Define 'RAG' in AI in one sentence.",
        "ground_truth": "RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval from external sources with LLM generation to produce more accurate and grounded responses."
    },
    {
        "id": 7,
        "category": "Reasoning",
        "question": "A warehouse receives 500 units daily and ships 450. How many units accumulate in 2 weeks (14 days)? Answer with just the number.",
        "ground_truth": "700"
    },
    {
        "id": 8,
        "category": "Reasoning",
        "question": "If Model A has 85% accuracy and Model B has 92% accuracy, what is the absolute improvement? Answer as a percentage.",
        "ground_truth": "7%"
    },
    {
        "id": 9,
        "category": "Classification",
        "question": "Classify this review as positive or negative: 'The product broke after one day. Terrible quality.' Answer: positive or negative.",
        "ground_truth": "negative"
    },
    {
        "id": 10,
        "category": "Classification",
        "question": "Classify this review as positive or negative: 'Amazing quality! Best purchase I made this year.' Answer: positive or negative.",
        "ground_truth": "positive"
    }
]

categories = Counter(t["category"] for t in TEST_SUITE)
pp({"total_questions": len(TEST_SUITE), "categories": dict(categories)}, "Test Suite Summary")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Evaluation Metrics</h2>
</div>

We'll implement three metrics at increasing levels of sophistication.

In [ ]:
# ============================================================
# 📏 Metric 1: Exact Match
# ============================================================

def exact_match(prediction: str, ground_truth: str) -> float:
    """Returns 1.0 if prediction matches ground truth (case-insensitive), 0.0 otherwise."""
    pred = prediction.strip().lower().rstrip('.')
    truth = ground_truth.strip().lower().rstrip('.')
    # Check if truth is contained in prediction (for short answers)
    if truth in pred or pred in truth:
        return 1.0
    return 0.0

# Test
print("Exact Match tests:")
print(f"  'Paris' vs 'Paris': {exact_match('Paris', 'Paris')}")
print(f"  'paris' vs 'Paris': {exact_match('paris', 'Paris')}")
print(f"  'The answer is Paris.' vs 'Paris': {exact_match('The answer is Paris.', 'Paris')}")
print(f"  'London' vs 'Paris': {exact_match('London', 'Paris')}")

In [ ]:
# ============================================================
# 📏 Metric 2: Simplified BLEU Score
# ============================================================

def simple_bleu(prediction: str, ground_truth: str) -> float:
    """Simplified BLEU score based on word overlap (unigram precision)."""
    pred_words = prediction.lower().split()
    truth_words = ground_truth.lower().split()

    if not pred_words or not truth_words:
        return 0.0

    # Count matching words
    truth_counts = Counter(truth_words)
    matches = 0
    for word in pred_words:
        if truth_counts.get(word, 0) > 0:
            matches += 1
            truth_counts[word] -= 1

    precision = matches / len(pred_words)

    # Brevity penalty
    bp = min(1.0, len(pred_words) / len(truth_words)) if len(truth_words) > 0 else 0

    return round(precision * bp, 4)

# Test
print("BLEU Score tests:")
print(f"  Identical: {simple_bleu('the cat sat on the mat', 'the cat sat on the mat')}")
print(f"  Similar:   {simple_bleu('the cat is on the mat', 'the cat sat on the mat')}")
print(f"  Different: {simple_bleu('dogs run in the park', 'the cat sat on the mat')}")

In [ ]:
# ============================================================
# 📏 Metric 3: Cosine Similarity via Embeddings
# ============================================================

def get_embedding(text: str) -> list:
    """Get OpenAI embedding for a text string."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(pred: str, truth: str) -> float:
    """Compute cosine similarity between embeddings of two texts."""
    emb1 = np.array(get_embedding(pred))
    emb2 = np.array(get_embedding(truth))
    return round(float(np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))), 4)

# Test
print("Cosine Similarity tests:")
print(f"  Identical meaning: {cosine_similarity('The dog is happy', 'The dog is joyful')}")
print(f"  Similar topic:     {cosine_similarity('Machine learning models', 'AI algorithms')}")
print(f"  Unrelated:         {cosine_similarity('The dog is happy', 'Supply chain management')}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Exact match is strict (good for factual Q&A), BLEU measures word overlap (good for generation), and cosine similarity captures <b>semantic meaning</b> (best for open-ended answers where wording varies but meaning should match).
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Run the Evaluation</h2>
</div>

Let's run both models through the test suite and score every answer.

In [ ]:
# ============================================================
# 🏃 Run Models Through Test Suite
# ============================================================

def evaluate_model(model_name: str, call_fn, test_suite: list) -> list:
    """Run a model through the test suite and compute all metrics."""
    results = []
    for test in test_suite:
        # Get model's answer
        answer = call_fn(test["question"])

        # Compute metrics
        em = exact_match(answer, test["ground_truth"])
        bleu = simple_bleu(answer, test["ground_truth"])
        cos_sim = cosine_similarity(answer, test["ground_truth"])

        results.append({
            "id": test["id"],
            "category": test["category"],
            "answer": answer[:100],
            "exact_match": em,
            "bleu": bleu,
            "cosine_sim": cos_sim
        })
        print(f"  Q{test['id']:2d} [{test['category']:14s}] EM={em:.0f} BLEU={bleu:.3f} Cos={cos_sim:.3f} → {answer[:60]}")

    return results

# GPT-4.1-mini
def gpt_call(question):
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0
    )
    return r.choices[0].message.content.strip()

print("🔵 Evaluating GPT-4.1-mini...\n")
gpt_results = evaluate_model("GPT-4.1-mini", gpt_call, TEST_SUITE)

# Gemini
if gemini_available:
    def gemini_call(question):
        r = gemini_model.generate_content(question)
        return r.text.strip()

    print("\n🟢 Evaluating Gemini...\n")
    gemini_results = evaluate_model("Gemini", gemini_call, TEST_SUITE)
else:
    gemini_results = None
    print("\n⚠️ Skipping Gemini (add GEMINI_API_KEY to compare)")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Visualize Results</h2>
</div>

In [ ]:
# ============================================================
# 📊 Visualization: Scores per Question
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d0d1a')

metrics = ['exact_match', 'bleu', 'cosine_sim']
titles = ['Exact Match', 'BLEU Score', 'Cosine Similarity']
questions = [f"Q{r['id']}" for r in gpt_results]
x = np.arange(len(questions))
width = 0.35

for i, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[i]
    ax.set_facecolor('#1a1a2e')

    gpt_scores = [r[metric] for r in gpt_results]
    bars1 = ax.bar(x - width/2, gpt_scores, width, label='GPT-4.1-mini', color='#4A90D9', alpha=0.85)

    if gemini_results:
        gem_scores = [r[metric] for r in gemini_results]
        bars2 = ax.bar(x + width/2, gem_scores, width, label='Gemini', color='#4ecdc4', alpha=0.85)

    ax.set_xlabel('Question', color='white', fontsize=11)
    ax.set_ylabel('Score', color='white', fontsize=11)
    ax.set_title(title, color='white', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(questions, color='white', fontsize=9)
    ax.tick_params(colors='white')
    ax.set_ylim(0, 1.1)
    ax.legend(facecolor='#1a1a2e', edgecolor='#333', labelcolor='white', fontsize=9)
    ax.grid(axis='y', alpha=0.2, color='white')

plt.tight_layout()
plt.savefig('eval_results.png', dpi=150, facecolor=fig.get_facecolor(), bbox_inches='tight')
plt.show()

# Summary stats
print("\n📊 Summary Statistics:")
print(f"\n🔵 GPT-4.1-mini:")
for metric in metrics:
    avg = np.mean([r[metric] for r in gpt_results])
    print(f"   {metric:15s}: {avg:.3f}")

if gemini_results:
    print(f"\n🟢 Gemini:")
    for metric in metrics:
        avg = np.mean([r[metric] for r in gemini_results])
        print(f"   {metric:15s}: {avg:.3f}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> No single metric tells the whole story. <b>Exact match</b> is strict (good for factual), <b>BLEU</b> catches partial credit, and <b>cosine similarity</b> captures semantic meaning. Production evaluation suites use all three plus domain-specific metrics.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "Which metric is best for evaluating open-ended responses where the exact wording may differ but the meaning should match?",
        "options": [
            "Exact match — it compares strings character by character",
            "BLEU score — it counts word overlap",
            "Cosine similarity of embeddings — it compares semantic meaning",
            "Token count — shorter answers are always better"
        ],
        "answer": 2,
        "explanation": "Cosine similarity computes how close two texts are in embedding space, capturing semantic meaning regardless of exact wording. 'The dog is happy' and 'The canine is joyful' have high cosine similarity despite having zero word overlap."
    },
    {
        "q": "Why is it important to test LLMs across multiple categories (factual, math, reasoning, classification)?",
        "options": [
            "To make the test longer and more impressive",
            "Because LLMs perform differently on different task types — a model strong at facts may be weak at reasoning",
            "To use more API credits",
            "Categories don't matter — one metric on one task is enough"
        ],
        "answer": 1,
        "explanation": "LLMs have different strengths across task types. A model might ace factual recall but struggle with multi-step reasoning. Testing across categories reveals these nuances and helps you choose the right model for YOUR specific use case."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Add Your Own Metric

Implement an **"LLM-as-Judge"** metric — use a separate LLM call to score how well the answer matches the ground truth. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: LLM-as-Judge Metric
# ============================================================
# Replace each ----- with the correct value

def llm_judge(prediction: str, ground_truth: str, question: str) -> float:
    """Use an LLM to judge how well the prediction answers the question."""
    response = client.chat.completions.create(
        model="-----",                    # Which model for judging?
        messages=[{
            "role": "user",
            "content": f"""Rate how well this answer responds to the question.

Question: {question}
Expected Answer: {ground_truth}
Actual Answer: {prediction}

Rate from 0.0 to 1.0 where:
- 1.0 = perfect answer, matches the expected meaning
- 0.5 = partially correct
- 0.0 = completely wrong

Reply with ONLY a number between 0.0 and 1.0, nothing else."""
        }],
        temperature=-----               # Low or high temperature for scoring?
    )
    try:
        score = float(response.choices[0].message.------.strip())  # Get the text content
        return min(max(score, 0.0), 1.0)  # Clamp to [0, 1]
    except ValueError:
        return 0.5  # Default if parsing fails

# Test it on a few examples
print("🧑‍⚖️ LLM-as-Judge tests:\n")
test_pairs = [
    ("Paris", "Paris", "What is the capital of France?"),
    ("The capital of France is Paris.", "Paris", "What is the capital of France?"),
    ("London", "Paris", "What is the capital of France?"),
]

for pred, truth, q in test_pairs:
    score = llm_judge(pred, truth, q)
    print(f"  Score: {score:.1f} | Pred: '{pred}' vs Truth: '{truth}'")

In [ ]:
# --- Expected Output ---
show_expected("Score: 1.0 for exact match (Paris vs Paris)\nScore: ~0.9 for paraphrased match\nScore: 0.0-0.2 for wrong answer (London vs Paris)")

**📝 Your Observations:** *(double-click to edit)*

1. Which question type does the model score lowest on? Why might that be? _[Your answer]_

2. Which metric correlated best with your intuition about answer quality? _[Your answer]_

3. Where did GPT and Gemini disagree most? What does that tell you about choosing models for specific tasks? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Expand the test suite:</b> Add 10 more questions in a new category (e.g., "Coding" or "Ethics")</li>
    <li><b>Per-category analysis:</b> Compute average scores per category and visualize as a radar chart</li>
    <li><b>Temperature sweep:</b> Run the same test suite at temperature 0, 0.5, and 1.0 — how do scores change?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built an LLM evaluation framework with 3 metrics, a 10-question test suite, and professional visualizations.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Exact match</b> — strict, binary. Best for factual/classification tasks</li>
      <li><b>BLEU score</b> — word overlap with brevity penalty. Good for generation tasks</li>
      <li><b>Cosine similarity</b> — semantic meaning via embeddings. Best for open-ended answers</li>
      <li><b>LLM-as-Judge</b> — use AI to evaluate AI. Most flexible but adds cost</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M12: SDKs & Claude Code Workshops</p>
</div>